# numerische Ableitung eines Zeitsignals

Dieses Notebook zeigt verschiedene Möglichkeiten in Python die numerische Ableitung (Differentation) eines Zeitsignals durch folgende Ansätze anzunähern:
- Vorwärtsdifferenz:  
$$
y_{k} = \frac{u_{i+1} - u_{i}}{\text{dT}}
$$

- Rückwärtsdifferenz: 
$$
y_{k} = \frac{u_{i}  - u_{i-1}}{\text{dT}}
$$

- Zentraldifferenz mit 2 Elementen: 
$$
y_{k} = \frac{\frac{1}{2}u_{i+1}  - \frac{1}{2}u_{i-1} }{ \text{dT}}
$$

- Zentraldifferenz mit 4 Elementen: 
$$
y_{k} = \frac{-\frac{1}{12}u_{i+2} ] + \frac{2}{3}u_{i+1}  - \frac{2}{3}u_{i-1}  + \frac{1}{12}u_{i-2} }{\text{dT}}
$$
mit Eingang $u_{k}$ und Ausgang $y_{k}$ zum Abtastschritt $k$ und der konstanten Abtastzeit $\text{dT}$.

Zur Berechnung werden folgenden Python Funktionen verwendet:

- Vorwärts-und Rückwärtzdifferenz mit `np.diff()`
- Finite difference coefficient
   - Berechnung über Faltung (Convolution) mit `np.convolve()`
   - Berechnung mit linearen Filtern  mit `scipy.signal.lfilter()`
   
Zum Schluß erfolgt ein einfaches Anwendungsbeispiel mit den Testsignalen:
- Sinus 
- Sinus mit Rauschen überlagert
- einem Sprung 

---
2026-06-07 ug V1.3

<!---
https://en.wikipedia.org/wiki/Five-point_stencil
https://en.wikipedia.org/wiki/Finite_difference_coefficient
--->

----
Python Packages laden

In [ ]:
import numpy as np
from matplotlib import pyplot as plt
%matplotlib inline

## Vorwärts-und Rückwärtzdifferenz mit `np.diff()`

Mit der Numpy-Funktion [`np.diff()`](https://numpy.org/doc/stable/reference/generated/numpy.diff.html)
werden die diskreten Differenzen zwischen den aufeinanderfolgenden Elementen einer Sequenz berechnet.

- Vorwärtsdifferenz (forward difference) $\text{out}[i] = \text{in}[i+1] - \text{in}[i]$
- Rückwärtsdifferent (backward difference) $\text{out}[i] = \text{in}[i] - \text{in}[i-1]$

Wir wollen zunächst verstehen, wie die  Numpy-Funktion [`np.diff()`](https://numpy.org/doc/stable/reference/generated/numpy.diff.html) arbeitet.

Dazu erzeugen wir uns einen Vektor mit einer aufsteigenden Reihe der Zahlen von 0 bis 9:

In [ ]:
x = np.arange(10)
x

und wenden die Funktion [`np.diff()`](https://numpy.org/doc/stable/reference/generated/numpy.diff.html) auf diesen Vektor an:

In [ ]:
diff_x = np.diff(x)
diff_x

In [ ]:
x.shape, diff_x.shape

`np.diff(x)` hat ein Element weniger als x !!!

Um auf einen gleich langen Ausgang zu kommen, kann mit den Parametern `prepend=` oder `append=` ein Wert an das Eingangsarray voran oder nach gestellt werden:

In [ ]:
diff_x = np.diff(x, prepend=-1)
diff_x

In [ ]:
diff_x = np.diff(x, append=10)
diff_x

In [ ]:
diff_x = np.diff(x, prepend=np.nan)
diff_x

In [ ]:
diff_x = np.diff(x, append=np.nan)
diff_x

---
## Finite difference coefficient

Die numersche Ableitung kann durch finite Differenzen angenähert werden, siehe
[Finite difference coefficient - Wikipedia](https://en.wikipedia.org/wiki/Finite_difference_coefficient).

Neben der oben bereits erwähnten
- Rückwärtsdifferent $\text{out}[i] = \text{in}[i] - \text{in}[i-1]$

gibt es auch:
- Zentraldifferenz mit 2 Elementen: $\text{out}[i] = \frac{1}{2}\text{in}[i+1] - \frac{1}{2}\text{in}[i-1]$
- Zentraldifferenz mit 4 Elementen: $\text{out}[i] = -\frac{1}{12}\text{in}[i+2] + \frac{2}{3}\text{in}[i+1] - \frac{2}{3}\text{in}[i-1] + \frac{1}{12}\text{in}[i-2]$.

Die Zentraldifferenzen greifen auf zukünftige Werte einer Sequenz zu. Diese Verfahren können nur angewendet werden, wenn die Messung als Ganzes im Speicher vorliegt, also als _Batch_, vorliegt.

Am Anfang oder am Ende der Sequenz kann es zu Einschwing- bzw. Ausschwingvorgänge kommen, wenn die Anfangs- und Endbedinungen nicht passen. 

#### Berechnung über Faltung (Convolution) mit `np.convolve()`

Mit der Numpy-Funktion [`np.convolve()`](https://numpy.org/doc/stable/reference/generated/numpy.convolve.html) wird die diskrete lineare Faltung zweier eindimensionaler Sequenzen berechnet.

Mit dem Parameter `mode=` wird bestimmt, wie die Faltung zurückgegeben wird. 
Mit  `mode='same'` ist die berechnete Sequenz so lange, wie die längere der beiden Eingangssequenzen. 

Um die Faltung und auch Berechnung der numerischenAbleitung besser zu verstehen, erzeugen wir uns ein _Einheitspuls_-Testsignal:

Die folgende Sequenz besteht aus 10 Nullen, an deren fünfter Position, die Null durch eine Eins ersetzt wurde:

In [ ]:
N = 10
test_sequenz = np.zeros(N)
test_sequenz[4] = 1
test_sequenz

Rückwärtsdifferent $\text{out}[i] = \text{in}[i] - \text{in}[i-1]$

In [ ]:
backward_difference_pattern = [1,-1]

Ableitung_1 = np.convolve(test_sequenz, backward_difference_pattern, mode='same')
print(f'Shape {Ableitung_1.shape}')
Ableitung_1

Zentraldifferenz mit 2 Elementen: $\text{out}[i] = \frac{1}{2}\text{in}[i+1] - \frac{1}{2}\text{in}[i-1]$

In [ ]:
central_difference_with_2_elements = [1/2,0,-1/2]

Ableitung_2=np.convolve(test_sequenz, central_difference_with_2_elements, mode='same')
print(f'Shape {Ableitung_2.shape}')
Ableitung_2

In [ ]:
central_difference_with_4_elements = [-1/12, 2/3, 0, -2/3, 1/12]

Ableitung_3=np.convolve(test_sequenz, central_difference_with_4_elements, mode='same')
print(f'Shape {Ableitung_3.shape}')
Ableitung_3

In [ ]:
def plot_sequence(ax,sequence,label):
    ax.plot(sequence,marker='o',linestyle='None',label=label)
    ax.vlines(range(N),np.maximum(sequence,0),np.minimum(sequence,0))
    ax.axhline(0)
    ax.grid(True)
    ax.set_ylim(-1.3,1.3)
    ax.legend()


# -----------------------------------

fig, (ax1,ax2,ax3,ax4) = plt.subplots(nrows=4, figsize=(8,10))

plot_sequence(ax1,test_sequenz,"Eingang")
plot_sequence(ax2,Ableitung_1,"Rückwärtsdifferent")
plot_sequence(ax3,Ableitung_2,"Zentraldifferenz 2 Elemente")
plot_sequence(ax4,Ableitung_3,"Zentraldifferenz 4 Elemente")

### Berechnung mit linearen Filtern mit `scipy.signal.lfilter()`

Mit der Funktion [`scipy.signal.lfilter()`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.lfilter.html) lassen sich lineare Filter durch Angabe einer zeitdiskreten Übertragungsfunktion bzw. Differenzengleichung berechnen.

$
    H(z) = \frac{B(z)}{A(z)} = \frac{b_0 + b_1 z^{-1} + b_2 z^{-2} + ... + b_p z^{-p}}{a_0 + a_1 z^{-1} + a_2 z^{-1} + ...+ b_q z^{-q}}
$

In [ ]:
import scipy.signal as signal

#### Backwarddifferenz 

$
    H(z) = \frac{B(z)}{A(z)} = \frac{1 - z^{-1}}{1}
$

In [ ]:
B = np.array([1.0, -1.0])
A = np.array([1])
diff_x = signal.lfilter(B,A,test_sequenz)
print(f'Shape {diff_x.shape}')
diff_x

#### Zentraldifferenz mit 2 Elementen

$
    H(z) = \frac{B(z)}{A(z)} = \frac{\frac{1}{2} - \frac{1}{2} z^{-2}}{1}
$

In [ ]:
B = np.array([1/2,0,-1/2])
A = np.array([1])
diff_x = signal.lfilter(B,A,test_sequenz)
print(f'Shape {diff_x.shape}')
diff_x

Die berechnete Sequenz ist gegenüber der, mit der `np.convolve()`-Funktion berechneten, Sequenz um einen Schritt verzögert.
Wir müssen die Sequenz, um einen Schritt nach links schieben.
Dazu können wir die Funktion [`np.roll()`](https://numpy.org/doc/stable/reference/generated/numpy.roll.html) verwenden.

In [ ]:
diff_x = np.roll(diff_x,-1)
diff_x[-1] = np.nan  # Werte als nicht bekannt markieren
diff_x

#### Zentraldifferenz mit 4 Elementen

$
    H(z) = \frac{B(z)}{A(z)} = \frac{-\frac{1}{12} + \frac{2}{3}z^{-1}  - \frac{2}{3} z^{-3}  + \frac{1}{12} z^{-4}}{1}
$

In [ ]:
B = np.array([-1/12, 2/3, 0, -2/3, 1/12])
A = np.array([1])
diff_x = signal.lfilter(B,A,test_sequenz)

# Sequenz um zwei Schritte nach links schieben
diff_x = np.roll(diff_x,-2)
diff_x[-2:] = np.nan  # Werte als nicht bekannt markieren
print(f'Shape {diff_x.shape}')
diff_x

----
### Beispiel: numerische Ableitung eines Sinussignals
#### Testsignalerzeugung
Wir bauen uns als ein weiteres Testsignal, diesmal ein Sinussignal mit 5Hz und Amplitude von 1

In [ ]:
N = 200     # Anzahl der Abtastpunkte
dT = 0.01   # Abtastzeit in Sekunden

# time signal
t = np.arange(0,N*dT,dT)

# 5 Hz sinus signal  (corner frequency; filter shall attentuate signal by 3dB = 0.707)
f0 = 5.0
Ampl = 1.0
input_signal = Ampl * np.sin(2*np.pi*f0*t)

# wir berechen die analytische Ableitung des 
derivative_input_signal =  Ampl * 2 * np.pi *f0 * np.cos(2*np.pi*f0*t)

#### Berechnung der numerischen Ableitung

numerische Ableitung mit 
- Vorwärtsdifferenz:  $\text{out}[i] = (\text{in}[i+1] - \text{in}[i]) / \text{dT}$
- Rückwärtsdifferenz: $\text{out}[i] = (\text{in}[i] - \text{in}[i-1])/ \text{dT}$

- Zentraldifferenz mit 2 Elementen: $\text{out}[i] = (\frac{1}{2}\text{in}[i+1] - \frac{1}{2}\text{in}[i-1])/ \text{dT}$
- Zentraldifferenz mit 4 Elementen: $\text{out}[i] = (-\frac{1}{12}\text{in}[i+2] + \frac{2}{3}\text{in}[i+1] - \frac{2}{3}\text{in}[i-1] + \frac{1}{12}\text{in}[i-2])/ \text{dT}$



mit $\text{dT}$ als Abtastzeit

In [ ]:
def calc_derivative(input_signal, dT=1.0, mode="backward_difference" ):
    """
    mode:
        "forward_difference"
        "backward_difference"
        "center_difference_2"
        "center_difference_4"
    """
    if mode == "backward_difference":
        #return np.diff(input_signal,prepend=np.nan)/dT
        return np.convolve(input_signal, [1,-1], mode='same')/dT
    elif mode == "forward_difference":
        return np.diff(input_signal,append=np.nan)/dT
    elif mode == "center_difference_2":
        return np.convolve(input_signal, [1/2,0,-1/2], mode='same')/dT
    elif mode == "center_difference_4":
        return np.convolve(input_signal, [-1/12, 2/3, 0, -2/3, 1/12], mode='same')/dT

In [ ]:
def plot_derivativ(t, input_signal,derivative_input_signal):
    
    dT = np.mean(np.diff(t))
    
    print(f"dT: {dT} s")
    
    fig,(ax1, ax2) = plt.subplots(nrows=2,figsize=(16,8),sharex=True)

    # Eingangssignal
    ax1.set_title('Berechnung numerischer Ableitungen')
    ax1.plot(t,input_signal,color='b',label='input',marker='o')
    ax1.set_ylabel('Eingangssignal')
    ax1.grid()
    
    
    # Ableitungen
    if derivative_input_signal is not None:
        ax2.plot(t,derivative_input_signal,color='c',label='Ableitung analytisch berechnet',marker='o',lw=3)

    ax2.plot(t,calc_derivative(input_signal,dT,"forward_difference"),color='r',label='Vorwärtsdifferenz',marker='o')
    ax2.plot(t,calc_derivative(input_signal,dT,"backward_difference"),color='m',label='Rückwärtsdifferenz',marker='o')
    ax2.plot(t,calc_derivative(input_signal,dT,"center_difference_2"),color='g',label='Zentraldifferenz 2',marker='o')
    ax2.plot(t,calc_derivative(input_signal,dT,"center_difference_4"),color='orange',label='Zentraldifferenz 4',marker='o')

    ax2.grid()
    ax2.set_xlabel('Zeit [s]')
    ax2.set_ylabel('Ableitung')
    ax2.legend()
    
    
    ax2.set_xlim((-0.01,0.3))
    
    

In [ ]:
plot_derivativ(t, input_signal,derivative_input_signal)

Anmerkung zum Vergleich der berechneten Ableitungen:
<br>
Die Ableitung, die über die Vortwärtsdifferenz berechnet wurde, eilt der analytisch berechneten Ableitung leicht voraus, während die Ableitung, die über die Rückwärtsdifferenz berechnet wurde, dieser etwas nacheilt. 
Die Ableitungen, die über die Zentraldifferenzen berechnet wurden, sind in Phase mit der analytisch berechneten Ableitung.

---
### Signale mit Rauschen überlagert

Rauschen raut die Ableitung auf.

In [ ]:
sigma_noise = 0.002
noise = np.random.normal(0, np.sqrt(sigma_noise), N)
input_signal_noise = input_signal + noise

In [ ]:
plot_derivativ(t, input_signal_noise,derivative_input_signal)

Das überlagerte Rauschen raut die Ableitungen, die über die Vorwärts und Rückwärtsdifferenzen berechnet wurden, stärker auf als die Ableitungen die über die Zentraldifferenzen berechnet wurden.

Die Zentraldifferenzen besitzen somit eine glättende Eigenschaft.

----
### Neues Testsignal: Sprung 

#### Sprung bei 0.1 Sekunden

In [ ]:
t_start_step = 0.1
  
input_signal2 = np.zeros_like(t)
input_signal2[t>0.1] = 1.0
derivative_input_signal2 = None

In [ ]:
plot_derivativ(t, input_signal2,derivative_input_signal2)

Wir setzen deutlich, dass die Vorwärts- und die Zentraldifferenz bereits vor dem Sprung am Eingang mit einer Antwort am Ausgang beginnen. Sie Sprungantworten sind nicht kausal, man bezeichnet die Filter auch als nicht kausal,
siehe [causal filter - Wikipedia](https://en.wikipedia.org/wiki/Causal_filter)